# Project 1 - Exploration and model selection

This notebook shows the steps that led to our final model:

1. data loading and exploration,
2. preprocessing (see `preprocessing.py`),
3. comparison of the six basic methods of `implementations.py` (baselines),
4. our model: class-weighted regularized logistic regression trained with Newton's method,
   with hyperparameters chosen by 5-fold cross-validation,
5. an ablation study of our modifications.

All the models are trained on 60% of the training set and evaluated on the other 40% (hold-out set).
The final predictions are produced by `run.py`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from helpers import load_csv_data, load_feature_names, split_data
from implementations import (
    mean_squared_error_gd,
    mean_squared_error_sgd,
    least_squares,
    ridge_regression,
    logistic_regression,
    reg_logistic_regression,
)
from preprocessing import Preprocessor
from models import (
    balanced_class_weights,
    newton_logistic_regression,
    predict_labels,
    accuracy,
    f1_score,
)
from cross_validation import (
    cross_validation_ridge,
    cross_validation_threshold,
    cross_validation_lambda_threshold,
)
from plots import plot_features, plot_cross_validation

## 1. Data loading and exploration

In [ ]:
x, _, y, _, _ = load_csv_data("data")
feature_names = load_feature_names("data")
print(f"{x.shape[0]} samples, {x.shape[1]} features")
print(f"Positive samples (y = 1): {np.mean(y == 1):.2%}")

The classes are very imbalanced (about 9% of positive samples): a model that always predicts -1 has an accuracy of 91% but an F1 score of 0. This is why we use the F1 score to compare models.

In [ ]:
nan_ratio = np.isnan(x).mean(axis=0)
plt.figure(figsize=(5, 3.5))
plt.hist(nan_ratio, bins=20)
plt.xlabel("fraction of missing values (NaN)")
plt.ylabel("number of features")
plt.show()
print(f"Features with >= 95% of NaN: {np.sum(nan_ratio >= 0.95)}")

In [ ]:
# Histogram of every feature (long to display)
# plot_features(x, feature_names)

## 2. Preprocessing

The preprocessing is learned on the training part only and then applied to the hold-out part (see `preprocessing.py` for the details of each step):
survey metadata removed, special BRFSS codes decoded (e.g. 88 = "none" -> 0, 77/99 = "don't know"/"refused" -> missing, 2xx = "times per month" -> times per day),
columns with too many missing values or almost constant removed, one-hot encoding of the columns with at most 10 values,
standardization, missing values -> 0, removal of highly correlated columns, and pairwise products of the 30 columns most correlated with the label.

In [ ]:
x_tr, x_va, y_tr, y_va = split_data(x, y, ratio=0.6, seed=42)
del x

preprocessor = Preprocessor()  # default values = final model
tx_tr = preprocessor.fit_transform(x_tr, feature_names, y_tr)
tx_va = preprocessor.transform(x_va)
print(f"Features: {x_tr.shape[1]} raw -> {tx_tr.shape[1]} after preprocessing")

## 3. Baselines: the six basic methods

Regression methods predict 1 if the score `tx @ w >= 0`. The logistic methods are trained with labels in {0, 1} and predict 1 if the probability is >= 0.5 (i.e. `tx @ w >= 0`).

In [ ]:
# The baselines use the cleaned features without interactions (some products are
# collinear, which makes the least-squares system singular).
baseline_prep = Preprocessor(n_interactions=0)
tb_tr = baseline_prep.fit_transform(x_tr, feature_names, y_tr)
tb_va = baseline_prep.transform(x_va)
d = tb_tr.shape[1]
y01_tr = (y_tr == 1).astype(float)  # labels in {0, 1} for the logistic methods
# Step sizes: gradient descent converges for gamma < 2/L (MSE) and gamma < 8/L
# (logistic loss), where L is the largest eigenvalue of tb_tr^T tb_tr / N.
np.random.seed(1)

ridge_lambdas = np.logspace(-4, 0, 7)
best_ridge_lambda, _ = cross_validation_ridge(y_tr, tb_tr, 5, ridge_lambdas)

baselines = {
    "MSE gradient descent": mean_squared_error_gd(y_tr, tb_tr, np.zeros(d), 100, 0.02),
    "MSE SGD": mean_squared_error_sgd(y_tr, tb_tr, np.zeros(d), 10000, 0.001),
    "Least squares": least_squares(y_tr, tb_tr),
    f"Ridge (lambda={best_ridge_lambda:.0e})": ridge_regression(y_tr, tb_tr, best_ridge_lambda),
    "Logistic regression": logistic_regression(y01_tr, tb_tr, np.zeros(d), 100, 0.1),
    "Reg. logistic regression": reg_logistic_regression(y01_tr, tb_tr, 1e-4, np.zeros(d), 100, 0.1),
}
for name, (w, _) in baselines.items():
    y_pred = predict_labels(tb_va, w)
    print(f"{name:28s} F1 = {f1_score(y_va, y_pred):.3f}   accuracy = {accuracy(y_va, y_pred):.3f}")
del tb_tr, tb_va

With the default threshold, all these models predict almost only -1 because of the class imbalance, so their F1 score is very low. We address this with two modifications:

* **balanced class weights**: each sample has weight N / (2 N_class), so both classes count equally in the loss;
* **a tuned decision threshold** on the score `tx @ w`, chosen to maximize the F1 score.

## 4. Our model: weighted regularized logistic regression (Newton's method)

### 4.1 Regularization and decision threshold (5-fold cross-validation, validation F1 score)

For each lambda, every threshold is evaluated; we keep the pair with the best mean validation F1 score.

In [ ]:
lambdas = np.logspace(-4, -2, 5)
thresholds = np.arange(-2, 2, 0.1)
best_lambda, best_threshold, f1_cv = cross_validation_lambda_threshold(
    y_tr, tx_tr, 5, lambdas, thresholds, max_iters=10
)
print(f"Best lambda: {best_lambda:.2e}, best threshold: {best_threshold:.1f} (cross-validation F1 = {f1_cv.max():.3f})")
plot_cross_validation(lambdas, f1_cv.max(axis=1), "lambda", "best validation F1 score")
plot_cross_validation(
    thresholds, f1_cv[list(lambdas).index(best_lambda)], "threshold on tx @ w", "validation F1 score", log_scale=False
)

### 4.2 Evaluation on the hold-out set

In [ ]:
w, _ = newton_logistic_regression(
    y_tr, tx_tr, np.zeros(tx_tr.shape[1]), 15, best_lambda, balanced_class_weights(y_tr)
)
y_pred = predict_labels(tx_va, w, best_threshold)
print(f"Hold-out set: F1 = {f1_score(y_va, y_pred):.3f}, accuracy = {accuracy(y_va, y_pred):.3f}")

The final model is retrained on the whole training set with these hyperparameters by `run.py`, which writes the submission file.

## 5. Ablation study

Each variant removes or adds one modification; the threshold is tuned by 5-fold cross-validation on the training part and the F1 score is measured on the hold-out set (this cell takes about 30 minutes).

In [ ]:
def evaluate(preprocessor, lambda_, weighted=True, tune_threshold=True):
    t_tr = preprocessor.fit_transform(x_tr, feature_names, y_tr)
    t_va = preprocessor.transform(x_va)
    threshold = 0.0
    if tune_threshold:
        threshold, _ = cross_validation_threshold(y_tr, t_tr, 5, lambda_, thresholds, 10, weighted)
    weights = balanced_class_weights(y_tr) if weighted else None
    w, _ = newton_logistic_regression(y_tr, t_tr, np.zeros(t_tr.shape[1]), 15, lambda_, weights)
    y_pred = predict_labels(t_va, w, threshold)
    del t_tr, t_va
    return t_tr.shape[1], threshold, f1_score(y_va, y_pred), accuracy(y_va, y_pred)

old = dict(drop_metadata=False, decode_codes=False, max_categories=5, n_interactions=0)
variants = {
    "Old preprocessing (first submission)": (Preprocessor(**old), 2.15e-5, True, True),
    "+ metadata removed, codes decoded": (Preprocessor(max_categories=5, n_interactions=0), best_lambda, True, True),
    "+ one-hot up to 10 categories": (Preprocessor(n_interactions=0), best_lambda, True, True),
    "+ 30 interactions (final)": (Preprocessor(), best_lambda, True, True),
    "Final without class weights": (Preprocessor(), best_lambda, False, True),
    "Final without threshold tuning": (Preprocessor(), best_lambda, True, False),
}
for name, (prep, lambda_, weighted, tune) in variants.items():
    d, t, f1, acc = evaluate(prep, lambda_, weighted, tune)
    print(f"{name:40s} D = {d:4d}  threshold = {t:5.1f}  F1 = {f1:.3f}  accuracy = {acc:.3f}")